# Exp1 & Exp2: film composition vs. sputtering context (NOMAD Oasis, read-only)

**Exp1** - relation between the *spatially averaged film composition* (EDX map average per library) and the *time-averaged DC bias* of each sputter target during a run.

**Exp2** - *oxygen atomic fraction* (EDX) vs. recorded deposition context: target material and sputtering base pressure.

(Exp3-6 are paused and intentionally not part of this notebook.)

## Hypotheses
- H1: a more negative / larger-magnitude target DC bias (more energetic sputtering, different target poisoning state) shifts the film composition of the target's elements; the sign and size of the trend is the question, not assumed.
- H2: higher base pressure (more residual H2O/O2) increases oxygen uptake; oxygen fraction may also depend on target material (oxygen affinity of the metal).

## Permissions and read-only behaviour
- Only `POST /entries/query` and `POST /entries/archive/query` are used, both with `owner: visible`: entries that *this account* is authorised to see (own + shared + published). This **cannot bypass user permissions**; unauthorised/private entries from other users are simply absent from the results, so counts may differ between users.
- The token is read from `NOMAD_CLIENT_ACCESS_TOKEN` or, on Windows, from the DPAPI-encrypted local store created by `setup_nomad_token.py`. It is never embedded, printed or committed. Without a token, the API only returns public data.
- Nothing is written back to NOMAD (no uploads, no `analysis.save()`, no files written by this notebook).

## Query limits
1. A small **metadata** query (entry ids only) with pagination, bounded by `PAGE_SIZE * MAX_PAGES` per schema.
2. **Archive** download bounded by `MAX_ARCHIVES`; EDX is parsed in small batches and its raw arrays discarded promptly, while linked library archives alone are fetched for joining. Raise the limits deliberately; the result is a *sample* unless the totals are covered (shown in the diagnostics).

## Assumptions (from the plugin schemas)
- `DTUSputtering`: `base_pressure` (stored in Pa, converted to Torr here); `steps[*].sources[*].vapor_source` holds the power-supply variant with `avg_dc_bias` and optionally a `dc_bias` time series (`value`, `time`); `deposition_parameters.<gun>` is a `SourceOverview` (`target_name`, `target_material`, `target_usage`); `substrates` are mountings; `samples` references the generated `DTUCombinatorialLibrary` entries (library `lab_id` = `<run lab_id>_<mounting name>`).
- `EDXMeasurement`: `results[*].quantifications[*].element` + `atomic_fraction`; `samples` references libraries. EDX is preferred over `DTUCombinatorialSample.composition` for measured spatial-map composition.
- Relations are **not** assumed one-to-one: a run has many libraries, a library may have several EDX entries, an EDX entry may reference several samples, and a run may have several targets. Ambiguous rows are flagged and excluded by default (configurable).
- Exp1 is joined only through the substantiated chain EDX -> library <- sputtering `samples`. If no link is found, an explicit blocker with diagnostics is shown; nothing is fabricated.
- Time-averaged bias = `avg_dc_bias` if present, else the mean of the `dc_bias` time series; averaging over steps is duration-weighted when durations are available, otherwise a plain mean (the basis is reported).

## Adapting to schema changes
Edit `SCHEMAS` (qualified names), the field names in `parse_sputtering` / `parse_edx`, and `resolve_ref`. Run the *raw payload inspection* cell to see the real archive structure first.

In [16]:
from pathlib import Path
import sys
for candidate in (Path.cwd(), Path.cwd() / 'notebooks' / 'oasis_trends'):
    if (candidate / 'oasis_auth.py').exists():
        sys.path.insert(0, str(candidate))
        break
else:
    raise FileNotFoundError('Could not find notebooks/oasis_trends/oasis_auth.py; start Jupyter in the repository or notebook folder.')

import os, re, math
import numpy as np
import pandas as pd
import requests
import plotly.express as px
import plotly.graph_objects as go
from oasis_auth import api_base_url, get_token

# ---------------- configuration ----------------
# Nanolab Oasis API (override with NOMAD_BASE_URL).
API_BASE_URL = api_base_url()  # NOMAD_BASE_URL, else NORTH's NOMAD_CLIENT_URL, else the Nanolab Oasis
OWNER = 'visible'  # entries the account is authorised to see; do not change to bypass anything

PKG = 'nomad_dtu_nanolab_plugin.schema_packages'
SCHEMAS = {
    'sputtering': f'{PKG}.sputtering.DTUSputtering',
    'edx': f'{PKG}.edx.EDXMeasurement',
    'library': f'{PKG}.sample.DTUCombinatorialLibrary',
}
EXTRA_QUERY = {}  # optional additional filters, e.g. {'upload_name': 'my upload'}

PAGE_SIZE = 50
MAX_PAGES = 60  # current visible metadata totals fit within this bound
MAX_ARCHIVES = {'sputtering': 1000, 'edx': 5000, 'library': 5000}  # fetch all current visible records/references
ARCHIVE_CHUNK = 10
TIMEOUT_S = 60

# analysis filters
ONLY_DEPOSITION_STEPS = True        # use only steps with creates_new_thin_film == True (if the field exists)
ALLOW_MULTI_SAMPLE_EDX = False      # EDX entries referencing >1 sample are ambiguous
ALLOW_MULTI_RUN_LIBRARY = False     # libraries linked from >1 sputtering run are ambiguous
MIN_EDX_POINTS = 3
O_FRACTION_RANGE = (0.0, 1.0)       # atomic fraction
BASE_PRESSURE_RANGE_TORR = None     # e.g. (1e-9, 1e-5) or None
PA_PER_TORR = 101325 / 760          # archive values are SI (Pa); pressures are reported in Torr
TARGET_FILTER = None                # e.g. ['Zr', 'Cu'] or None
EXP1_Y_ELEMENTS = ['auto', 'O']     # 'auto' = the target's own element if target_material is an element symbol

# ---------------- session (environment or encrypted Windows store) ----------------
SESSION = requests.Session()
_token = get_token()
if _token:
    SESSION.headers['Authorization'] = f'Bearer {_token}'
del _token
print('API base URL:', API_BASE_URL)
print('Token found in NOMAD_CLIENT_ACCESS_TOKEN:', 'Authorization' in SESSION.headers)
if 'Authorization' not in SESSION.headers:
    print('WARNING: no token; only public entries will be visible.')


def post(path, payload):
    r = SESSION.post(f'{API_BASE_URL}{path}', json=payload, timeout=TIMEOUT_S)
    if not r.ok:
        raise RuntimeError(f'HTTP {r.status_code} {r.reason} for {path}: {r.text[:300]}')
    return r.json()


def list_entries(qualified_name):
    """Paginated metadata-only query (owner: visible). Returns (rows, total)."""
    query = {'section_defs.definition_qualified_name:all': [qualified_name], **EXTRA_QUERY}
    rows, after, total = [], None, None
    for _ in range(MAX_PAGES):
        pagination = {'page_size': PAGE_SIZE, 'order_by': 'entry_id', 'order': 'asc'}
        if after:
            pagination['page_after_value'] = after
        res = post('/entries/query', {
            'owner': OWNER,
            'query': query,
            'required': {'include': ['entry_id', 'upload_id', 'entry_name', 'entry_type', 'mainfile', 'entry_create_time']},
            'pagination': pagination,
        })
        rows += res.get('data', [])
        total = res.get('pagination', {}).get('total')
        after = res.get('pagination', {}).get('next_page_after_value')
        if not after:
            break
    return rows, total


def fetch_archives(entry_ids):
    """Bounded archive download (owner: visible). Returns {entry_id: archive['data']}."""
    out = {}
    for i in range(0, len(entry_ids), ARCHIVE_CHUNK):
        part = list(entry_ids[i:i + ARCHIVE_CHUNK])
        res = post('/entries/archive/query', {
            'owner': OWNER,
            'query': {'entry_id:any': part},
            'required': {'data': '*'},
            'pagination': {'page_size': len(part)},
        })
        for d in res.get('data', []):
            out[d['entry_id']] = (d.get('archive') or {}).get('data') or {}
    return out


API base URL: https://nomad.nanolab.dtu.dk/nomad-oasis/api/v1
Token found in NOMAD_CLIENT_ACCESS_TOKEN: True


## 1. Small metadata query with pagination
Counts are for the entries visible to this account. `total` is the server-side count, `listed` is what is retrieved under the page bound.

In [17]:
meta, summary = {}, []
for key, qn in SCHEMAS.items():
    rows, total = list_entries(qn)
    meta[key] = pd.DataFrame(rows, columns=['entry_id', 'upload_id', 'entry_name', 'entry_type', 'mainfile', 'entry_create_time']) if not rows else pd.DataFrame(rows)
    summary.append({'schema_key': key, 'qualified_name': qn, 'total_visible': total, 'listed': len(rows)})
pd.DataFrame(summary)


,schema_key,qualified_name,total_visible,listed
0,sputtering,nomad_dtu_nanolab_plugin.schema_packages.sputt...,246,246
1,edx,nomad_dtu_nanolab_plugin.schema_packages.edx.E...,986,986
2,library,nomad_dtu_nanolab_plugin.schema_packages.sampl...,2078,2078


In [18]:
def ids_of(key):
    df = meta.get(key)
    return [] if df is None or df.empty else df['entry_id'].tolist()


upload_of = {}
mainfile_to_entry = {}
for df in meta.values():
    for r in df.to_dict('records'):
        upload_of[r['entry_id']] = r.get('upload_id')
        mainfile_to_entry[(r.get('upload_id'), r.get('mainfile'))] = r['entry_id']

archives = {}
for key in ('sputtering',):
    ids = ids_of(key)[:MAX_ARCHIVES[key]]
    archives[key] = fetch_archives(ids)
    print(f'{key}: requested {len(ids)} archives, received {len(archives[key])} (listed {len(ids_of(key))})')


sputtering: requested 246 archives, received 246 (listed 246)


## 2. Inspect the actual sputtering payload (raw structure, no assumptions)
Shows the structure of a fetched sputtering archive. EDX archives are streamed through the parser below rather than retained in memory.

In [19]:
def shape(obj, depth=0, max_depth=3):
    if depth >= max_depth:
        return type(obj).__name__
    if isinstance(obj, dict):
        return {k: shape(v, depth + 1, max_depth) for k, v in list(obj.items())[:25]}
    if isinstance(obj, list):
        return [f'list[{len(obj)}]', shape(obj[0], depth + 1, max_depth) if obj else None]
    return type(obj).__name__


import pprint
for key, arch in archives.items():
    if arch:
        eid = next(iter(arch))
        print(f'=== {key} example entry {eid} ===')
        pprint.pprint(shape(arch[eid]), width=110, compact=True)
    else:
        print(f'=== {key}: no archives fetched ===')


=== sputtering example entry -0vj6vfCG4xmrNNCudUDs9a_-Gck ===
{'base_pressure': 'float',
 'datetime': 'str',
 'deposition_parameters': {'ar_flow': 'float',
                           'deposition_temperature': 'float',
                           'deposition_temperature_2': 'float',
                           'deposition_temperature_setpoint': 'float',
                           'deposition_time': 'float',
                           'deposition_true_temperature': 'float',
                           'h2s_in_ar_flow': 'float',
                           'h2s_partial_pressure': 'float',
                           'magkeeper3': {'applied_power': 'float',
                                          'average_voltage': 'float',
                                          'end_voltage': 'float',
                                          'max_voltage': 'float',
                                          'min_voltage': 'float',
                                          'power_type': 'str',
            

## 3. Defensive parsers and reference resolution

In [20]:
def num(x):
    try:
        v = float(x)
    except (TypeError, ValueError):
        return None
    return v if math.isfinite(v) else None


def fnum(x):
    v = num(x) if not isinstance(x, (list, dict)) else None
    return np.nan if v is None else v


def as_list(x):
    return [] if x is None else (x if isinstance(x, list) else [x])


def norm(s):
    return re.sub('[^a-z0-9]', '', str(s).lower())


ENTRY_RE = re.compile(r'/archive/([A-Za-z0-9_-]{20,})')
ENTRY_API_RE = re.compile(r'/entries/([A-Za-z0-9_-]{20,})/archive')
MAINFILE_UP_RE = re.compile(r'/uploads/([^/]+)/archive/mainfile/([^#]+)')
MAINFILE_RE = re.compile(r'/archive/mainfile/([^#]+)')


def resolve_ref(ref, upload_id):
    """Resolve a NOMAD reference string to an entry_id (None if not resolvable)."""
    if not isinstance(ref, str):
        return None
    m = MAINFILE_UP_RE.search(ref)
    if m:
        return mainfile_to_entry.get((m.group(1), m.group(2)))
    m = MAINFILE_RE.search(ref)
    if m:
        return mainfile_to_entry.get((upload_id, m.group(1)))
    m = ENTRY_RE.search(ref) or ENTRY_API_RE.search(ref)
    return m.group(1) if m else None


def series_stats(dc):
    """(mean, duration_s, n_points) of a dc_bias time series dict; None where unavailable."""
    if not isinstance(dc, dict):
        return None, None, 0
    v = np.array([fnum(a) for a in as_list(dc.get('value'))], dtype=float)
    t = np.array([fnum(a) for a in as_list(dc.get('time'))], dtype=float)
    mean = float(np.nanmean(v)) if v.size and np.isfinite(v).any() else None
    dur = float(np.nanmax(t) - np.nanmin(t)) if t.size > 1 and np.isfinite(t).any() else None
    return mean, dur, int(v.size)


def parse_sputtering(eid, data, entry_create_time=None):
    dp = data.get('deposition_parameters') or {}
    guns = {g: d for g, d in dp.items() if isinstance(d, dict) and any(k in d for k in ('target_material', 'target_name', 'target_usage'))}
    steps = as_list(data.get('steps'))
    gun_targets = sorted({d.get('target_material') for d in guns.values() if d.get('target_material')})
    run = {
        'run_entry_id': eid, 'run_lab_id': data.get('lab_id'), 'run_name': data.get('name'),
        'base_pressure_torr': (lambda v: None if v is None else v / PA_PER_TORR)(num(data.get('base_pressure'))), 'entry_create_time': entry_create_time, 'datetime': data.get('datetime'), 'n_steps': len(steps),
        'n_substrates': len(as_list(data.get('substrates'))), 'n_sample_refs': len(as_list(data.get('samples'))),
        'overview_guns': ','.join(guns), 'overview_targets': '+'.join(gun_targets),
    }
    bias_rows = []
    for si, step in enumerate(steps):
        if not isinstance(step, dict):
            continue
        for ci, src in enumerate(as_list(step.get('sources'))):
            if not isinstance(src, dict):
                continue
            vs = src.get('vapor_source') if isinstance(src.get('vapor_source'), dict) else {}
            avg = num(vs.get('avg_dc_bias'))
            ts_mean, dur, npts = series_stats(vs.get('dc_bias'))
            text = norm(' '.join(str(x) for x in (src.get('name'), vs.get('name')) if x))
            gun = next((g for g in guns if norm(g) and norm(g) in text), None)
            if gun:
                ov = guns[gun]
                target, tname, usage, basis = ov.get('target_material'), ov.get('target_name'), num(ov.get('target_usage')), 'overview gun name match'
            else:
                names = [m.get('name') or m.get('lab_id') for m in as_list(src.get('material')) if isinstance(m, dict)]
                target, tname, usage = ('+'.join(n for n in names if n) or None), None, None
                basis = 'source.material' if target else 'unmatched'
            bias_rows.append({
                'run_entry_id': eid, 'step_idx': si, 'step_name': step.get('name'),
                'creates_new_thin_film': step.get('creates_new_thin_film'),
                'source_idx': ci, 'source_name': src.get('name'), 'ps_m_def': vs.get('m_def'),
                'avg_dc_bias_v': avg, 'ts_mean_v': ts_mean, 'ts_points': npts, 'duration_s': dur,
                'bias_v': avg if avg is not None else ts_mean,
                'bias_basis': 'avg_dc_bias' if avg is not None else ('time series mean' if ts_mean is not None else 'missing'),
                'target_material': target, 'target_name': tname,
                'target_usage_kwh': usage,
                'target_match': basis,
            })
    links = []
    for s in as_list(data.get('samples')):
        if isinstance(s, dict):
            links.append({'run_entry_id': eid, 'sample_ref': s.get('reference'), 'sample_lab_id': s.get('lab_id'),
                          'library_entry_id': resolve_ref(s.get('reference'), upload_of.get(eid))})
    return run, bias_rows, links


def parse_edx(eid, data):
    samples = [s for s in as_list(data.get('samples')) if isinstance(s, dict)]
    points = []
    for res in as_list(data.get('results')):
        comp = {}
        for q in as_list(res.get('quantifications') if isinstance(res, dict) else None):
            if isinstance(q, dict) and q.get('element') and num(q.get('atomic_fraction')) is not None:
                comp[q['element']] = num(q.get('atomic_fraction'))
        points.append(comp)
    summ = {'edx_entry_id': eid, 'n_samples': len(samples), 'n_points': len(points)}
    df = pd.DataFrame(points)
    if not df.empty:
        sums = df.sum(axis=1, skipna=True)
        scale = 100.0 if sums.median() > 1.5 else 1.0  # percent -> fraction
        df = df / scale
        summ.update({'sum_median_raw': float(sums.median()), 'scaled_from_percent': scale == 100.0})
        for el in df.columns:
            summ[f'frac_{el}'] = float(df[el].mean())
            summ[f'npts_{el}'] = int(df[el].notna().sum())
    links = [{'edx_entry_id': eid, 'n_samples': len(samples), 'sample_ref': s.get('reference'), 'sample_lab_id': s.get('lab_id'),
              'library_entry_id': resolve_ref(s.get('reference'), upload_of.get(eid))} for s in samples]
    return summ, links


meta_by_id = {r['entry_id']: r for df in meta.values() for r in df.to_dict('records')}
runs, bias_rows, run_links = [], [], []
for eid, data in archives.get('sputtering', {}).items():
    r, b, l = parse_sputtering(eid, data, meta_by_id.get(eid, {}).get('entry_create_time'))
    runs.append(r); bias_rows += b; run_links += l
edx_summ, edx_links = [], []
# Process EDX archives in small batches and discard raw payloads promptly;
# EDX result arrays can make retaining every archive exceed notebook memory.
edx_ids = ids_of('edx')[:MAX_ARCHIVES['edx']]
for start in range(0, len(edx_ids), 2):
    edx_batch = fetch_archives(edx_ids[start:start + 2])
    for eid, data in edx_batch.items():
        s, l = parse_edx(eid, data)
        edx_summ.append(s); edx_links += l
    del edx_batch
print(f'edx: parsed {len(edx_summ)} of {len(edx_ids)} listed archives')

runs = pd.DataFrame(runs)
bias = pd.DataFrame(bias_rows)
run_links = pd.DataFrame(run_links, columns=['run_entry_id', 'sample_ref', 'sample_lab_id', 'library_entry_id'])
edx_comp = pd.DataFrame(edx_summ)
edx_links = pd.DataFrame(edx_links, columns=['edx_entry_id', 'n_samples', 'sample_ref', 'sample_lab_id', 'library_entry_id'])
print('runs:', len(runs), '| bias rows:', len(bias), '| run->sample links:', len(run_links))
print('EDX entries:', len(edx_comp), '| EDX->sample links:', len(edx_links))


edx: parsed 986 of 986 listed archives
runs: 246 | bias rows: 1615 | run->sample links: 1123
EDX entries: 986 | EDX->sample links: 981


## 4. Libraries (bounded fetch, referenced libraries first) and diagnostics

In [21]:
referenced = [e for e in pd.concat([run_links['library_entry_id'], edx_links['library_entry_id']]).dropna().unique()]
# Only linked library archives are needed for joins; avoid downloading every
# unrelated library archive. NOMAD still applies owner: visible.
to_fetch = referenced[:MAX_ARCHIVES['library']]
archives['library'] = fetch_archives(to_fetch)
lib_lab_by_entry = {e: d.get('lab_id') for e, d in archives['library'].items()}
print(f'libraries: fetched {len(archives["library"])} linked archives (bound {MAX_ARCHIVES["library"]}; {len(referenced)} linked IDs identified)')


def add_key(df):
    """Join key: library lab_id (from the reference or the fetched library), else the entry id."""
    df = df.copy()
    def key(r):
        if isinstance(r['sample_lab_id'], str) and r['sample_lab_id']:
            return r['sample_lab_id']
        lab = lib_lab_by_entry.get(r['library_entry_id'])
        if lab:
            return lab
        return f'entry:{r["library_entry_id"]}' if isinstance(r['library_entry_id'], str) else None
    df['lib_key'] = df.apply(key, axis=1) if len(df) else pd.Series(dtype=object)
    return df


run_links, edx_links = add_key(run_links), add_key(edx_links)
diag = {
    'run->sample refs': len(run_links),
    'run->sample refs without lib_key': int(run_links['lib_key'].isna().sum()) if len(run_links) else 0,
    'EDX->sample refs': len(edx_links),
    'EDX->sample refs without lib_key': int(edx_links['lib_key'].isna().sum()) if len(edx_links) else 0,
    'EDX entries with 0 sample refs': int((edx_comp['n_samples'] == 0).sum()) if len(edx_comp) else 0,
    'EDX entries with >1 sample refs': int((edx_comp['n_samples'] > 1).sum()) if len(edx_comp) else 0,
    'EDX entries without any element quantification': int(len([c for c in edx_comp.columns if c.startswith('frac_')]) == 0) if len(edx_comp) else 0,
    'libraries shared by EDX and a sputtering run': len(set(run_links['lib_key'].dropna()) & set(edx_links['lib_key'].dropna())),
    'runs without base_pressure': int(runs['base_pressure_torr'].isna().sum()) if len(runs) else 0,
    'bias rows (all)': len(bias),
    'bias rows with a bias value': int(bias['bias_v'].notna().sum()) if len(bias) else 0,
    'bias rows without target match': int((bias['target_match'] == 'unmatched').sum()) if len(bias) else 0,
}
display(pd.Series(diag, name='count').to_frame())
if len(bias):
    display(bias.groupby(['ps_m_def', 'bias_basis'], dropna=False).size().rename('rows').reset_index())
if len(edx_comp) and 'frac_O' in edx_comp:
    print('EDX entries with oxygen quantified:', int(edx_comp['frac_O'].notna().sum()), 'of', len(edx_comp))
elif len(edx_comp):
    print('No EDX entry reports oxygen.')


libraries: fetched 1390 linked archives (bound 5000; 1390 linked IDs identified)


,count
run->sample refs,1123
run->sample refs without lib_key,0
EDX->sample refs,981
EDX->sample refs without lib_key,0
EDX entries with 0 sample refs,5
EDX entries with >1 sample refs,0
EDX entries without any element quantification,0
libraries shared by EDX and a sputtering run,683
runs without base_pressure,41
bias rows (all),1615


,ps_m_def,bias_basis,rows
0,nomad_dtu_nanolab_plugin.schema_packages.sputt...,missing,4
1,nomad_dtu_nanolab_plugin.schema_packages.sputt...,avg_dc_bias,1246
2,NaN,missing,365


EDX entries with oxygen quantified: 930 of 986


## 5. Join tables (explicit stage counts; many-to-many kept visible)

In [22]:
# Per run x target time-averaged bias (duration-weighted if all durations present, else plain mean)
def agg_bias(g):
    g = g[g['bias_v'].notna()]
    if g.empty:
        return pd.Series({'bias_v_mean': np.nan, 'n_steps_used': 0, 'weighting': 'none'})
    if g['duration_s'].notna().all() and g['duration_s'].sum() > 0:
        return pd.Series({'bias_v_mean': np.average(g['bias_v'], weights=g['duration_s']), 'n_steps_used': len(g), 'weighting': 'duration'})
    return pd.Series({'bias_v_mean': g['bias_v'].mean(), 'n_steps_used': len(g), 'weighting': 'plain mean'})


b = bias.copy()
if len(b) and ONLY_DEPOSITION_STEPS:
    if b['creates_new_thin_film'].notna().any():
        b = b[b['creates_new_thin_film'] == True]
    else:
        print('creates_new_thin_film missing in payload: cannot restrict to deposition steps (all steps used).')
run_target_bias = (b.groupby(['run_entry_id', 'target_material'], dropna=True).apply(agg_bias).reset_index()
                   if len(b) else pd.DataFrame(columns=['run_entry_id', 'target_material', 'bias_v_mean', 'n_steps_used', 'weighting']))
run_targets = (run_target_bias.groupby('run_entry_id')['target_material'].agg(lambda s: '+'.join(sorted(set(s)))).rename('targets').reset_index()
               if len(run_target_bias) else pd.DataFrame(columns=['run_entry_id', 'targets']))
runs_ctx = runs.merge(run_targets, on='run_entry_id', how='left') if len(runs) else runs
if len(runs_ctx):
    runs_ctx['targets'] = runs_ctx['targets'].fillna(runs_ctx['overview_targets'].replace('', np.nan))

# EDX -> library (one row per EDX x sample reference)
edx_lib = edx_links.merge(edx_comp.drop(columns=['n_samples']), on='edx_entry_id', how='left') if len(edx_links) and len(edx_comp) else pd.DataFrame()
if len(edx_lib):
    if not ALLOW_MULTI_SAMPLE_EDX:
        edx_lib = edx_lib[edx_lib['n_samples'] == 1]
    edx_lib = edx_lib[edx_lib['lib_key'].notna() & (edx_lib['n_points'] >= MIN_EDX_POINTS)]

run_lib = run_links[['run_entry_id', 'lib_key']].dropna().drop_duplicates() if len(run_links) else pd.DataFrame(columns=['run_entry_id', 'lib_key'])
run_lib['n_runs_for_lib'] = run_lib.groupby('lib_key')['run_entry_id'].transform('nunique') if len(run_lib) else []
if len(run_lib) and not ALLOW_MULTI_RUN_LIBRARY:
    run_lib = run_lib[run_lib['n_runs_for_lib'] == 1]

stages = {'EDX rows usable (single sample, enough points)': len(edx_lib), 'run->library links usable': len(run_lib)}
joined = edx_lib.merge(run_lib, on='lib_key', how='inner') if len(edx_lib) and len(run_lib) else pd.DataFrame()
stages['EDX x run joined rows'] = len(joined)
if len(joined):
    joined = joined.merge(runs_ctx[['run_entry_id', 'run_lab_id', 'base_pressure_torr', 'targets', 'entry_create_time']], on='run_entry_id', how='left')
display(pd.Series(stages, name='rows').to_frame())
if joined.empty:
    print('BLOCKER: no EDX entry could be joined to a sputtering run through a library reference.')
    print('Check: (a) the diagnostics above (shared libraries = 0?), (b) raise MAX_ARCHIVES/MAX_PAGES so both sides are covered,')
    print('(c) the reference format printed in the payload inspection / resolve_ref, (d) whether libraries are linked only via parent_library or process references.')
    display(edx_links.head(10)[['edx_entry_id', 'sample_ref', 'sample_lab_id', 'lib_key']] if len(edx_links) else edx_links)
    display(run_links.head(10)[['run_entry_id', 'sample_ref', 'sample_lab_id', 'lib_key']] if len(run_links) else run_links)


,rows
"EDX rows usable (single sample, enough points)",980
run->library links usable,1095
EDX x run joined rows,689


## 6. Exp2 - oxygen atomic fraction vs. target material and base pressure

In [23]:
exp2 = joined.copy()
if len(exp2) and 'frac_O' in exp2:
    n0 = len(exp2)
    exp2 = exp2[exp2['frac_O'].notna() & exp2['frac_O'].between(*O_FRACTION_RANGE)]
    if BASE_PRESSURE_RANGE_TORR:
        exp2 = exp2[exp2['base_pressure_torr'].between(*BASE_PRESSURE_RANGE_TORR)]
    if TARGET_FILTER:
        exp2 = exp2[exp2['targets'].apply(lambda t: isinstance(t, str) and any(x in t.split('+') for x in TARGET_FILTER))]
    print(f'Exp2 rows after filters: {len(exp2)} of {n0} joined rows')
    print('missing base_pressure:', int(exp2['base_pressure_torr'].isna().sum()), '| missing target:', int(exp2['targets'].isna().sum()))
    display(exp2[['edx_entry_id', 'lib_key', 'run_lab_id', 'targets', 'base_pressure_torr', 'frac_O', 'n_points']].head(20))

    p = exp2[exp2['base_pressure_torr'].notna()]
    if len(p):
        display(px.scatter(p, x='base_pressure_torr', y='frac_O', color='targets',
                           hover_data=['run_lab_id', 'lib_key'], log_x=(p['base_pressure_torr'] > 0).all(),
                           labels={'base_pressure_torr': 'Base pressure (Torr)', 'frac_O': 'Oxygen atomic fraction (EDX map mean)'},
                           title='Exp2: oxygen fraction versus base pressure').show())
    display(px.box(exp2, x='targets', y='frac_O', color='targets', points='all',
                   labels={'targets': 'Target material(s)', 'frac_O': 'Oxygen atomic fraction'},
                   title='Exp2: oxygen fraction by target').show())
else:
    print('Exp2 cannot be computed: no joined rows or no oxygen quantification in the fetched EDX entries (see diagnostics).')


Exp2 rows after filters: 639 of 689 joined rows
missing base_pressure: 90 | missing target: 0


,edx_entry_id,lib_key,run_lab_id,targets,base_pressure_torr,frac_O,n_points
0,-0Xx3lX-5GiYKg2EsBaVUJOVg3vH,chmode_0033_Zr_BR,chmode_0033_Zr,Zr_T_001,4.037565e-07,0.299767,18
1,-1ddbXdq8bjXbjXB6aOJe0M4LMNs,chmode_0001_Zn-Zr_BL,chmode_0001_Zn-Zr,Zn_T_001+Zr_T_001,2.569751e-07,0.064467,24
2,-IWWMQi9fPS0eSA1hHT-Bvy_EN0b,chmode_0031_Zr_BR,chmode_0031_Zr,Zr_T_001,3.297243e-07,0.214938,24
3,-S3xy9XJ9IMJr7G2J09HochetJ9R,mittma_0051_Cu_FR,mittma_0051_Cu,Cu_T_001,NaN,0.014417,24
4,-VzOTSW0ge3XnYBqZ-LgS_bJSOUx,eugbe_0015_Sb_FR,eugbe_0015_Sb,S+Sb_T_001,1.490000e-07,0.045233,21
5,-XOBXc5P3HKSzvGxGRRTORr6je3Y,chmode_0001_Zn-Zr_FL,chmode_0001_Zn-Zr,Zn_T_001+Zr_T_001,2.569751e-07,0.089540,25
6,-_vVZtPX1FKRw52B_ykoiBJqfHlf,chmode_0035_Zn_FL,chmode_0035_Zn,Zn_T_001,1.766067e-07,0.202430,20
7,-c-5b1bwWyvYUmOoSCbidZNJbDxU,mittma_0004_Cu_BR,mittma_0004_Cu,Untracked Material,1.109531e-07,0.247441,87
8,-jRr4qdZNsuLRZ-pNSSiBupMlVo4,mittma_0034_Cu_BL,mittma_0034_Cu,Cu_T_001,6.000000e-07,0.006276,25
9,-nfSZBsGaRbXCe13gUlWyPx_w9Pk,mittma_0017_Cu_FL,mittma_0017_Cu,Cu_T_001+S,2.500000e-07,0.081767,15


None

None

## 6b. Does oxygen change after a Zr-target deposition?
Compare EDX oxygen averages from films deposited before the first Zr run with those from later non-Zr runs. The Zr-target run itself is excluded because its base pressure was recorded before that deposition. Run order uses NOMAD entry creation timestamps; this is a coarse chronology, not a timestamp from the sputtering log.

In [30]:
CHRONOLOGY_START_DATE = '2025-10-01'  # Set to 'YYYY-MM-DD' to exclude earlier runs; None keeps all dates.
run_chronology = runs[['run_entry_id', 'datetime', 'overview_targets', 'base_pressure_torr']].copy()
run_chronology['run_time'] = pd.to_datetime(run_chronology['datetime'], utc=True, errors='coerce')
run_chronology = run_chronology.dropna(subset=['run_time']).drop_duplicates('run_entry_id')
run_chronology = run_chronology.sort_values(['run_time', 'run_entry_id']).reset_index(drop=True)
if CHRONOLOGY_START_DATE is not None:
    start_date = pd.to_datetime(CHRONOLOGY_START_DATE, utc=True, errors='raise')
    excluded_runs = int((run_chronology['run_time'] < start_date).sum())
    run_chronology = run_chronology[run_chronology['run_time'] >= start_date].copy()
    print(f'Chronology cutoff: {start_date.date()} (excluded {excluded_runs} earlier runs)')
run_chronology['is_zr_run'] = run_chronology['overview_targets'].fillna('').str.contains(
    r'(?<![A-Za-z])Zr(?![a-z])|zirconium', case=False, regex=True)
run_chronology['zr_sputtered_before'] = run_chronology['is_zr_run'].shift(fill_value=False).cummax()
if run_chronology.empty:
    print('No sputtering runs remain in the chronology after applying the date cutoff.')
elif run_chronology['is_zr_run'].any():
    zirconium_runs = run_chronology[run_chronology['is_zr_run']]
    print('Zr-target runs identified:', len(zirconium_runs))
    display(zirconium_runs[['run_time', 'run_entry_id', 'overview_targets', 'base_pressure_torr']])
    if len(exp2):
        oxygen_history = exp2.merge(
            run_chronology[['run_entry_id', 'run_time', 'is_zr_run', 'zr_sputtered_before']],
            on='run_entry_id', how='left')
        # Reduce multiple mapped samples to one mean per run to avoid treating
        # repeated samples from a deposition as independent run observations.
        oxygen_by_run = oxygen_history.groupby('run_entry_id', as_index=False).agg(
            run_time=('run_time', 'first'), oxygen_fraction=('frac_O', 'mean'),
            target_set=('targets', 'first'), is_zr_run=('is_zr_run', 'first'),
            zr_sputtered_before=('zr_sputtered_before', 'first'))
        non_zr_runs = ~oxygen_by_run['is_zr_run'].fillna(False).astype(bool)
        oxygen_by_run = oxygen_by_run[non_zr_runs].dropna(subset=['oxygen_fraction', 'run_time'])
        oxygen_by_run['period'] = oxygen_by_run['zr_sputtered_before'].map(
            {False: 'Before first Zr run', True: 'After a prior Zr run'})
        display(oxygen_by_run.groupby('period')['oxygen_fraction'].agg(['count', 'median', 'mean', 'std']))
        display(px.scatter(oxygen_by_run, x='run_time', y='oxygen_fraction', color='period',
                           hover_data=['run_entry_id', 'target_set'],
                           title='EDX oxygen by deposition chronology relative to Zr sputtering').show())
        display(px.box(oxygen_by_run, x='period', y='oxygen_fraction', points='all',
                       title='EDX oxygen before versus after a prior Zr-target run').show())
    else:
        print('No oxygen-bearing joined EDX/run pairs in the selected records.')
else:
    print('No Zr-target sputtering run identified in the fetched visible run archives; check target names and archive coverage.')

Chronology cutoff: 2025-10-01 (excluded 111 earlier runs)
Zr-target runs identified: 44


,run_time,run_entry_id,overview_targets,base_pressure_torr
171,2026-04-07 09:40:14.525000+00:00,Q4B7JRyYUJ_k-32i9BD-wGlzPKtU,Zn+Zr,2.569751e-07
173,2026-04-09 10:00:19.993000+00:00,06rgMLXoNEbxcVuKh34aGBuuM-2w,Zn+Zr,1.913116e-07
174,2026-04-10 10:22:54.943000+00:00,Ba6dKYmzORMfD0ckQSHXpIhnuFc-,Zr,1.851695e-07
175,2026-04-15 09:16:27.966000+00:00,0JuFlNcoZ9euCRW9crsvCdTHk9kR,Zr,1.751601e-07
177,2026-04-23 08:31:25.959000+00:00,6d-GiKo1M7V3xNDrlAau9hdekzP9,Zn+Zr,1.791910e-07
178,2026-04-23 13:42:18.760000+00:00,FyozWP6SWbvLAVTEq_dHTp73ZFMg,Zn+Zr,3.228036e-07
179,2026-04-24 07:24:25.936000+00:00,GSaz33FS9KR5vuqZlYTDmp5depgk,Zn+Zr,2.111597e-07
180,2026-04-24 13:14:04.385000+00:00,jZI1zXAH_AejhEKMxQtPBs38bwkr,Zn+Zr,2.085145e-07
181,2026-04-28 08:43:59.340000+00:00,MLadv0nzKbCO1Ayei1ULZRCzRk6a,Zr,2.299746e-07
182,2026-04-28 13:36:17.519000+00:00,UejSftHREndPfaPjQErNqfQ13FOg,Zn+Zr,2.145447e-07


,count,median,mean,std
period,,,,
After a prior Zr run,19,0.011888,0.047066,0.101339
Before first Zr run,55,0.037398,0.077598,0.086691


None

None

## 7. Exp1 - spatially averaged composition vs. time-averaged target DC bias
Per library, the EDX map means are averaged over its EDX entries; each library inherits the bias of every target of its (unique) sputtering run. A different library on the same run therefore shares the same bias value (not independent samples).

In [ ]:
frac_cols = [c for c in joined.columns if c.startswith('frac_')] if len(joined) else []
exp1 = pd.DataFrame()
if len(joined) and len(run_target_bias):
    lib_comp = joined.groupby(['lib_key', 'run_entry_id'])[frac_cols].mean().reset_index()
    lib_comp['n_edx'] = joined.groupby(['lib_key', 'run_entry_id']).size().values
    exp1 = lib_comp.merge(run_target_bias.dropna(subset=['bias_v_mean']), on='run_entry_id', how='inner')
    if TARGET_FILTER:
        exp1 = exp1[exp1['target_material'].isin(TARGET_FILTER)]
print('Exp1 rows (library x run x target with bias):', len(exp1))

if exp1.empty:
    print('BLOCKER for Exp1: no substantiated sample->sputtering link with a bias value.')
    print('Needed: EDX.samples -> library <- DTUSputtering.samples, and steps[*].sources[*].vapor_source.avg_dc_bias or dc_bias.value.')
    print('See the diagnostics tables above; no composition/bias pairing was fabricated.')
else:
    display(exp1[['lib_key', 'run_entry_id', 'target_material', 'bias_v_mean', 'weighting', 'n_steps_used', 'n_edx']].head(20))
    elem_re = re.compile(r'[A-Z][a-z]?')
    for spec in EXP1_Y_ELEMENTS:
        targets = sorted(exp1['target_material'].dropna().unique())[:8]
        plot_rows = []
        for tgt in targets:
            el = tgt if spec == 'auto' else spec
            col = f'frac_{el}'
            g = exp1[(exp1['target_material'] == tgt)]
            if (spec == 'auto' and not elem_re.fullmatch(str(tgt))) or col not in g:
                continue
            g = g[g[col].notna()]
            if not g.empty:
                g = g.copy()
                g['element_fraction'] = g[col]
                g['element'] = el
                plot_rows.append(g)
        plot_data = pd.concat(plot_rows, ignore_index=True) if plot_rows else pd.DataFrame()
        if plot_data.empty:
            print(f'No quantified {spec} composition rows for the available targets.')
            continue
        el_label = 'target element' if spec == 'auto' else spec
        fig = px.scatter(plot_data, x='bias_v_mean', y='element_fraction',
                         color='target_material', hover_data=['lib_key', 'run_entry_id', 'n_edx'],
                         labels={'bias_v_mean': 'Time-averaged DC bias (V)', 'element_fraction': 'Atomic fraction'},
                         title=f'Exp1: {el_label} composition versus time-averaged target bias')
        for tgt, group in plot_data.groupby('target_material'):
            element = tgt if spec == 'auto' else spec
            group = group.dropna(subset=['bias_v_mean', 'element_fraction'])
            if len(group) >= 3 and group['bias_v_mean'].nunique() > 1:
                coeffs = np.polyfit(group['bias_v_mean'], group['element_fraction'], 1)
                xs = np.linspace(group['bias_v_mean'].min(), group['bias_v_mean'].max(), 20)
                fig.add_trace(go.Scatter(x=xs, y=np.polyval(coeffs, xs), mode='lines',
                                         line={'dash': 'dash'}, name=f'{tgt} linear fit', showlegend=False))
        fig.show()


Exp1 rows (library x run x target with bias): 827


,lib_key,run_entry_id,target_material,bias_v_mean,weighting,n_steps_used,n_edx
0,albper_0002_Ba_BL,66lF4wUAlhTi0VjlNnysJLxIMUgg,Ba_T_001,104.043719,duration,1,1
1,albper_0002_Ba_BR,66lF4wUAlhTi0VjlNnysJLxIMUgg,Ba_T_001,104.043719,duration,1,1
2,albper_0002_Ba_FL,66lF4wUAlhTi0VjlNnysJLxIMUgg,Ba_T_001,104.043719,duration,1,1
3,albper_0002_Ba_FR,66lF4wUAlhTi0VjlNnysJLxIMUgg,Ba_T_001,104.043719,duration,1,1
4,albper_0003_Ba_BL,8BCRe_jEMOkV52BSoV56aAFf7fdc,Ba_T_001,98.469516,duration,1,1
5,albper_0003_Ba_BR,8BCRe_jEMOkV52BSoV56aAFf7fdc,Ba_T_001,98.469516,duration,1,1
6,albper_0003_Ba_FL,8BCRe_jEMOkV52BSoV56aAFf7fdc,Ba_T_001,98.469516,duration,1,1
7,albper_0003_Ba_FR,8BCRe_jEMOkV52BSoV56aAFf7fdc,Ba_T_001,98.469516,duration,1,1
8,albper_0004_Ba_BL,3r6CQjuhITN_NGbYv3OKTuw2BXrA,Ba_T_001,97.901135,duration,1,1
9,albper_0004_Ba_BR,3r6CQjuhITN_NGbYv3OKTuw2BXrA,Ba_T_001,97.901135,duration,1,1


No quantified auto composition rows for the available targets.


## Notes and known limitations
- `section_defs.definition_qualified_name:all` matches entries whose archive contains that section definition; if your Oasis uses another plugin package path, change `SCHEMAS` (or use `entry_type`).
- Only entries inside the metadata/archive bounds are analysed; compare `total_visible` with the fetched counts before drawing conclusions.
- Linking via `DTUCombinatorialLibrary.parent_library` / process references is not followed; add it if libraries are cleaved from parents (EDX on child pieces would then need the parent's run).
- Correlation here is exploratory (runs share bias values across libraries, small n, bias and pressure co-vary with target and run date); no results are written back to NOMAD.